# Lesson 7: Getting Structured Output

## JSON, Lists, and Specific Formats

In this lesson, you'll learn:
- How to get AI to respond in specific formats
- Working with JSON output
- Parsing AI responses in your code
- Build a Recipe Extractor that outputs structured data!

**Structured output = AI responses your code can easily work with!**


---

## Part 1: Why Structured Output?

### The Problem with Free-Form Text

```
┌─────────────────────────────────────────────────────────────────┐
│ You: "Give me info about Python"                                │
│                                                                 │
│ AI: "Python is a versatile programming language created by      │
│      Guido van Rossum in 1991. It's known for its simple        │
│      syntax and is used in web development, data science,       │
│      machine learning, and more..."                             │
│                                                                 │
│ Problem: How do you extract specific data from this?            │
│ - What year was it created? (Need to parse text)                │
│ - Who created it? (Need to find the name)                       │
│ - What is it used for? (Need to identify the list)              │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│ BETTER: Ask for JSON!                                           │
│                                                                 │
│ AI returns:                                                     │
│ {                                                               │
│   "name": "Python",                                             │
│   "creator": "Guido van Rossum",                                │
│   "year": 1991,                                                 │
│   "uses": ["web development", "data science", "machine learning"]│
│ }                                                               │
│                                                                 │
│ Now you can easily access: data["creator"] → "Guido van Rossum" │
└─────────────────────────────────────────────────────────────────┘
```


---

## Part 2: Setup


In [ ]:
# Setup - Run this first!

from dotenv import load_dotenv
from openai import OpenAI
import json  # We'll need this to parse JSON!

load_dotenv(override=True)
client = OpenAI()

def chat(prompt, temperature=0.7):
    """Basic chat function"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

print("✅ Ready to learn about structured output!")


---

## Part 3: Requesting JSON Output

The key is asking clearly for JSON in your prompt!


In [ ]:
# Basic JSON request

prompt = """
Give me information about the planet Mars.

Respond ONLY with valid JSON in this exact format:
{
    "name": "planet name",
    "type": "planet type",
    "distance_from_sun_km": number,
    "fun_fact": "one interesting fact"
}
"""

response = chat(prompt, temperature=0)
print("Raw AI Response:")
print(response)


In [ ]:
# Parse the JSON and use it in Python!

# Sometimes AI wraps JSON in markdown code blocks, so we clean it
def clean_json(text):
    """Remove markdown code blocks if present"""
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()

# Parse the JSON
cleaned = clean_json(response)
data = json.loads(cleaned)

print("Parsed Data:")
print(f"  Planet: {data['name']}")
print(f"  Type: {data['type']}")
print(f"  Distance from Sun: {data['distance_from_sun_km']:,} km")
print(f"  Fun Fact: {data['fun_fact']}")


---

## Part 4: Using JSON Mode

OpenAI has a built-in JSON mode that guarantees valid JSON output!


In [ ]:
# Using OpenAI's JSON mode

def get_json(prompt):
    """Get guaranteed valid JSON from AI"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}  # Enable JSON mode!
    )
    return json.loads(response.choices[0].message.content)

# Test it
prompt = """
Create a profile for a fictional superhero.
Include: name, powers (list of 3), weakness, and origin_city.
Respond in JSON format.
"""

hero = get_json(prompt)
print("Superhero Profile:")
print(json.dumps(hero, indent=2))


In [ ]:
# Now we can easily use the data!
print(f"\nHero Name: {hero['name']}")
print(f"Powers: {', '.join(hero['powers'])}")
print(f"Weakness: {hero['weakness']}")
print(f"From: {hero['origin_city']}")


---

## Part 5: Project - Recipe Extractor

Let's build a tool that extracts structured recipe data from any recipe description!


In [ ]:
# PROJECT: Recipe Extractor

def extract_recipe(description):
    """
    Extract structured recipe data from a text description.
    
    Args:
        description: Free-form text describing a recipe
        
    Returns:
        Structured recipe data as a dictionary
    """
    prompt = f"""
    Extract recipe information from the following description and return it as JSON.
    
    Description: {description}
    
    Return JSON with this structure:
    {{
        "name": "recipe name",
        "prep_time_minutes": number,
        "cook_time_minutes": number,
        "servings": number,
        "difficulty": "easy/medium/hard",
        "ingredients": [
            {{"item": "ingredient name", "amount": "quantity"}}
        ],
        "steps": ["step 1", "step 2", ...]
    }}
    """
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

print("✅ Recipe Extractor ready!")


In [ ]:
# Test with a recipe description

recipe_text = """
My grandma's famous chocolate chip cookies! You need 2 cups of flour, 
1 cup of butter, 1 cup of sugar, 2 eggs, and a bag of chocolate chips. 
Mix the butter and sugar, add eggs, then flour, then chips. Bake at 350°F 
for about 12 minutes. Makes around 24 cookies. Pretty easy to make, 
takes about 15 minutes to prep.
"""

recipe = extract_recipe(recipe_text)
print("📜 EXTRACTED RECIPE")
print("=" * 50)
print(json.dumps(recipe, indent=2))


In [ ]:
# Now display it nicely!

def display_recipe(recipe):
    """Display recipe in a nice format"""
    print(f"🍳 {recipe['name'].upper()}")
    print("=" * 50)
    print(f"⏱️  Prep: {recipe['prep_time_minutes']} min | Cook: {recipe['cook_time_minutes']} min")
    print(f"👥 Servings: {recipe['servings']} | Difficulty: {recipe['difficulty']}")
    print()
    print("📝 INGREDIENTS:")
    for ing in recipe['ingredients']:
        print(f"   • {ing['amount']} {ing['item']}")
    print()
    print("👨‍🍳 STEPS:")
    for i, step in enumerate(recipe['steps'], 1):
        print(f"   {i}. {step}")

display_recipe(recipe)


---

## Summary: Key Takeaways

### Getting Structured Output

1. **Be explicit** - Tell AI exactly what format you want
2. **Provide examples** - Show the exact JSON structure
3. **Use JSON mode** - `response_format={"type": "json_object"}`
4. **Low temperature** - Use 0 for consistent formatting

### The Pattern

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    response_format={"type": "json_object"}  # Guarantee JSON!
)
data = json.loads(response.choices[0].message.content)
```

### What You Built
A **Recipe Extractor** that converts messy text into clean, structured data!

### What's Next?
In **Lesson 8**, you'll learn about **Multi-Step Workflows** - chaining AI calls together!

**Excellent! Move on to Lesson 8!**
